In [2]:
# !pip install streamlit

import pandas as pd
import numpy as np
dataset_raw = pd.read_csv('Motor_Vehicle_Collisions_-_Crashes.csv', low_memory=False)
pd.set_option('display.max_columns', None)

# print(dataset_raw.info())
# print(dataset_raw.head())
# print(dataset_raw.shape)
dataset_clean = dataset_raw.copy()

dataset_clean['CRASH DATE'] = pd.to_datetime(dataset_clean['CRASH DATE'], format='%m/%d/%Y')
dataset_clean['MONTH'] = dataset_clean['CRASH DATE'].dt.month
wochentag_woerterbuch = {
    0: 'Monday',
    1: 'Tuesday',
    2: 'Wednesday',
    3: 'Thursday',
    4: 'Friday',
    5: 'Saturday',
    6: 'Sunday'
}
dataset_clean['WEEKDAY'] = dataset_clean['CRASH DATE'].dt.weekday.map(wochentag_woerterbuch).astype('string')
dataset_clean['CRASH HOUR'] = pd.to_datetime(dataset_clean['CRASH TIME'], format='%H:%M').dt.hour

dataset_clean['NUMBER OF PEOPLE INJURED'] = dataset_clean['NUMBER OF PERSONS INJURED'].fillna(0).astype(int)
dataset_clean['NUMBER OF PEOPLE KILLED'] = dataset_clean['NUMBER OF PERSONS KILLED'].fillna(0).astype(int)

# Erst als String-Datentyp deklarieren
text_spalten = [
    'BOROUGH',
    'VEHICLE TYPE CODE 1',
    'VEHICLE TYPE CODE 2',
    'VEHICLE TYPE CODE 3', 
    'VEHICLE TYPE CODE 4',
    'VEHICLE TYPE CODE 5',
    'CONTRIBUTING FACTOR VEHICLE 1',
    'CONTRIBUTING FACTOR VEHICLE 2', 
    'CONTRIBUTING FACTOR VEHICLE 3',
    'CONTRIBUTING FACTOR VEHICLE 4', 
    'CONTRIBUTING FACTOR VEHICLE 5'
]
dataset_clean[text_spalten] = dataset_clean[text_spalten].astype('string')

# 🧼 AUTOMATISCHE TEXT-BEREINIGUNG FÜR ALLE TEXTSPALTEN
for col in text_spalten:
    # 1. Text in Großbuchstaben umwandeln und Leerzeichen entfernen
    dataset_clean[col] = dataset_clean[col].str.upper().str.strip()
    
    # 2. 'UNSPECIFIED', 'UNKNOWN' und die numerischen Codes durch pd.NA (None) ersetzen
    unerwuenschte_werte = ['UNSPECIFIED', 'UNKNOWN', '1', '35', '36', '37', '52', '80']
    dataset_clean[col] = dataset_clean[col].replace(unerwuenschte_werte, pd.NA)


dataset_clean['ZIP CODE'] = dataset_clean['ZIP CODE'].replace(r'^\s*$', np.nan, regex=True)
dataset_clean['ZIP CODE'] = dataset_clean['ZIP CODE'].astype('Int64')

# --- SICHERE KOORDINATEN-BEREINIGUNG ---
# 1. Definiere die Grenzen für New York City
lat_min, lat_max = 40.49, 40.92
lon_min, lon_max = -74.26, -73.69

# 2. Finde Zeilen, die DEFINITIV außerhalb von NYC liegen
invalid_lat = (dataset_clean['LATITUDE'] < lat_min) | (dataset_clean['LATITUDE'] > lat_max)
invalid_lon = (dataset_clean['LONGITUDE'] < lon_min) | (dataset_clean['LONGITUDE'] > lon_max)

# 3. Setze NUR diese Ausreißer auf NaN (Originale NaN-Werte bleiben einfach NaN)
dataset_clean.loc[invalid_lat | invalid_lon, ['LATITUDE', 'LONGITUDE']] = np.nan
# ----------------------------------------



column_order = [
    'CRASH DATE',
    'MONTH',
    'WEEKDAY',
    'CRASH HOUR',
    'BOROUGH',
    'ZIP CODE',
    'LATITUDE',
    'LONGITUDE',
    'NUMBER OF PEOPLE INJURED',
    'NUMBER OF PEOPLE KILLED',
    'VEHICLE TYPE CODE 1',
    'VEHICLE TYPE CODE 2',
    'VEHICLE TYPE CODE 3',
    'VEHICLE TYPE CODE 4',
    'VEHICLE TYPE CODE 5',
    'CONTRIBUTING FACTOR VEHICLE 1',
    'CONTRIBUTING FACTOR VEHICLE 2',
    'CONTRIBUTING FACTOR VEHICLE 3',
    'CONTRIBUTING FACTOR VEHICLE 4',
    'CONTRIBUTING FACTOR VEHICLE 5',
]
dataset_clean = dataset_clean[column_order].fillna(pd.NA)
dataset_clean.to_csv('NYC_Collisions_Clean.csv', index=False)



print("\n\n\n")
print("=====dataset.info==============================================================================================")
print(dataset_clean.info())

print("\n\n\n")
print("=====dataset.head==============================================================================================")
print(dataset_clean.head(20))

print("\n\n\n")
print("=====dataset.describe==============================================================================================")
print(dataset_clean.describe())

print("\n\n\n")
print("=====dataset.shape==============================================================================================")
print(dataset_clean.shape)







=====dataset.info==============================================================================================
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2262089 entries, 0 to 2262088
Data columns (total 20 columns):
 #   Column                         Dtype         
---  ------                         -----         
 0   CRASH DATE                     datetime64[ns]
 1   MONTH                          int32         
 2   WEEKDAY                        string        
 3   CRASH HOUR                     int32         
 4   BOROUGH                        string        
 5   ZIP CODE                       Int64         
 6   LATITUDE                       float64       
 7   LONGITUDE                      float64       
 8   NUMBER OF PEOPLE INJURED       int64         
 9   NUMBER OF PEOPLE KILLED        int64         
 10  VEHICLE TYPE CODE 1            string        
 11  VEHICLE TYPE CODE 2            string        
 12  VEHICLE TYPE CODE 3            string        
 13  V